# Broadcast Pipeline — 4-Phase Camera ID Colab Guide

This notebook is a **complete walkthrough** of the broadcast video processing pipeline with the **4-phase camera ID** stack:
from raw `.mp4` input to per-camera on-screen text timelines (scoreboard graphics, logos, etc.).

**Related notebooks:** [`broadcast_pipeline_colab_guide.ipynb`](broadcast_pipeline_colab_guide.ipynb) (legacy 9-stage guide) · [`phase_a_scene_cluster_guide.ipynb`](phase_a_scene_cluster_guide.ipynb) (camera-ID-only deep dive)

## Table of contents

1. [Learning outcomes](#learning-outcomes)
2. [Pipeline architecture](#pipeline-architecture)
3. [4-phase camera ID](#4-phase-camera-id)
4. [Prerequisites](#prerequisites)
5. [Runtime profiles (free vs Pro)](#runtime-profiles)
6. [Environment setup](#environment-setup)
7. [Configuration](#configuration)
8. [Stage-by-stage deep dive](#stage-by-stage-deep-dive)
9. [Chunked execution](#chunked-execution)
10. [Human-in-the-loop reference editing](#human-in-the-loop)
11. [Full end-to-end run](#full-end-to-end-run)
12. [Results exploration](#results-exploration)
13. [Optional timeline web UI](#optional-timeline-web-ui)
14. [Standalone OCR debug](#standalone-ocr-debug)
15. [Troubleshooting](#troubleshooting)
16. [Out of scope](#out-of-scope)

## Learning outcomes

After working through this notebook you will understand:

1. **The problem** — turn a sports broadcast into structured per-camera text timelines.
2. **The workflow** — twelve ordered stages orchestrated by `run_pipeline()` in `src/broadcast_pipeline/orchestrator.py`.
3. **The 4-phase camera ID stack** — scene clustering, geometry ensemble, cut-boundary linking, and graphics fingerprint refine.
4. **The artifacts** — every CSV/JSON/JPEG written to disk, and how to resume or partially re-run.
5. **Colab constraints** — memory, session timeouts, GPU vs CPU trade-offs, and Drive persistence.

The CLI entry point `main.py` is equivalent to calling `run_pipeline(PipelineConfig(...))` programmatically.
**Do not** run `!python main.py` on Colab without fixing `sys.path` — `main.py` only adds `src/`, but inner modules import `from src.camera_assignemnt...` which requires the **repo root** on the path (see setup below).


## Pipeline architecture

```mermaid
flowchart TD
    video[Input video MP4] --> meta[meta: probe FPS/duration]
    meta --> extract[extract: scene cuts + frame sampling]
    extract --> filter[filter: full_court vs closeup]
    filter --> cameras[cameras: scene_cluster + geom_pose]
    cameras --> link[link: cut-boundary matching]
    link --> graphics_sparse[graphics_sparse: bbox detect]
    graphics_sparse --> ocr[ocr: RapidOCR + readability]
    ocr --> reference[reference: discover approved text]
    reference --> enrich[enrich: temporal IoU stabilization]
    enrich --> graphics_profile[graphics_profile: layout refine]
    graphics_profile --> associate[associate: map OCR to reference]
    associate --> aggregate[aggregate: per-camera timelines]
    aggregate --> outputs[CSVs + pipeline_summary.json]
```

**Stage order** (`STAGE_ORDER` in `src/broadcast_pipeline/config.py`):

`meta → extract → filter → cameras → link → graphics_sparse → ocr → reference → enrich → graphics_profile → associate → aggregate`

### Expected outputs

| Artifact | Stage | Purpose |
|----------|-------|---------|
| `video_meta.json` | meta | FPS, duration, resolution, frame count |
| `scenes.json` | extract | Scene cut boundaries (frame + time) |
| `frame_index.csv` | extract | Sampled frames with `camera` / `ocr` roles |
| `frames/*.jpg` | extract | Extracted frame JPEGs |
| `scene_types.csv` | filter | `full_court` vs `closeup` per scene — **gates cameras** |
| `scene_assignments.csv` | cameras, link, graphics | Camera ID per scene (+ `scene_type`, `assignment_source`) |
| `frame_assignments.csv` | cameras, link, graphics | Camera ID per sampled frame |
| `geom_pose_features.csv` | cameras (Phase B) | Per-scene homography pose debug |
| `cut_boundary_matches.csv` | link (Phase C) | Cut-boundary match scores |
| `frame_graphics_sparse.csv` | graphics_sparse (Phase D) | Detection-only graphics boxes |
| `frame_ocr.csv` | ocr | Raw OCR words + detections per frame |
| `approved_text_reference.csv` | reference | User-editable approved text list |
| `frame_ocr_enriched.csv` | enrich | Temporally stabilized OCR |
| `camera_graphics_profile.json` | graphics_profile (Phase D) | Per-camera layout fingerprints |
| `scene_assignments_graphics_refined.csv` | graphics_profile (Phase D) | Optional refine audit copy |
| `frame_text_associated.csv` | associate | OCR mapped to reference text |
| `dropped_text.csv` | associate | Tokens that failed association |
| `aggregated_complete.csv` | aggregate | Per-camera complete-text timeline |
| `aggregated_partial.csv` | aggregate | Per-camera partial-text timeline |
| `pipeline_summary.json` | aggregate | Run statistics |


## 4-phase camera ID

Camera identity is built in four incremental phases. Each phase maps to specific pipeline stages.

| Phase | Stages | What it does |
|-------|--------|--------------|
| **A** | `filter` + `cameras` | Scene-level clustering (`scene_cluster`): aggregate camera frames per scene, stratify `full_court` vs `closeup`, scene-timeline temporal fill |
| **B** | `cameras` | `geom_pose` fourth ensemble member — homography-derived viewpoint for `full_court` shots |
| **C** | `link` | LightGlue static-region matching at cuts; propagates wide `camera_id` → closeup |
| **D** | `graphics_sparse` + `graphics_profile` | Sparse bbox layout fingerprint + OCR-driven refine for residual unknowns |

Plans: [`phase_a`](../plans/phase_a_visual_clustering_foundations.md) · [`phase_b`](../plans/phase_b_geometry_ensemble.md) · [`phase_c`](../plans/phase_c_cut_boundary_linking.md) · [`phase_d`](../plans/phase_d_graphics_fingerprint.md)

```mermaid
flowchart LR
  fi[frame_index.csv]
  st[scene_types.csv]
  agg[aggregate per scene]
  strat[stratify full_court closeup]
  ens[ensemble HSV ResNet DINO geom_pose]
  stf[scene_temporal_fill]
  sa[scene_assignments.csv]
  link[link cut_boundary]
  gfx[graphics refine]
  sa --> link --> gfx
  fi --> agg --> strat --> ens --> stf --> sa
  st --> strat
```

**Default approach:** `camera_approach="scene_cluster"` in `PipelineConfig`. The legacy `embedding_cluster` path (per-frame majority vote) remains for comparison but is not used by default.


## Prerequisites

### Hardware / Colab runtime

| Requirement | Free tier | Colab Pro GPU |
|-------------|-----------|---------------|
| Runtime | CPU OK for Phase A (HSV) + text stages | T4/A100 for full ensemble + link + graphics |
| RAM | ≥12 GB; use free profile | ≥16 GB for ResNet50+DINOv2+geom_pose+LightGlue |
| Disk | Mount Google Drive; outputs include many JPEGs | Same |
| Session length | Use chunked cells + `resume=True` | Can often run end-to-end |

### Software

- **Python:** `pyproject.toml` declares `>=3.13`. Colab may ship 3.10–3.12. Install packages directly via pip in the setup cell below — **avoid** `pip install -e .` if it rejects your Python version.
- **System packages:** None required beyond pip wheels (OpenCV, PySceneDetect, etc.).
- **Network:** First run downloads torchvision weights (Pro), DINOv2, LightGlue weights, RapidOCR ONNX models (~hundreds of MB total, cached per session).

### Import path (critical)

Inner modules use `from src.camera_assignemnt...` and `from src.scene_ocr...`. You must add **both** the repo root and `src/` to `sys.path`:

```python
ROOT = Path("/content/CV_problem")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
```

### Project acquisition (upload zip)

1. Zip the project locally. **Exclude** `data/`, `graphify-out/`, `.git/` to keep size small.
2. Upload the zip to Colab Files or Google Drive.
3. Unzip via the setup cell (checks for `src/broadcast_pipeline/`, not just an empty folder).
4. Set `ROOT` and `ZIP_PATH` in the configuration / unzip cells to match your Drive layout.

#### Zip manifest

| Path | Required | Purpose |
|------|----------|---------|
| `src/broadcast_pipeline/` | Yes | Pipeline orchestration |
| `src/scene_ocr/` | Yes | OCR stage |
| `src/camera_assignemnt/scene_cluster/` | Yes | Phase A cameras |
| `src/camera_assignemnt/scene_classifier/` | Yes | Scene type filter |
| `src/camera_assignemnt/homography/` | Pro (Phase B) | `geom_pose` ensemble member |
| `src/camera_assignemnt/cut_match/` | Pro (Phase C) | Cut-boundary linker |
| `scripts/download_lightglue_weights.py` | Pro (Phase C) | Pre-cache matcher weights |
| `static/timeline_viz/` | Optional | Timeline web UI |
| `main.py` | Optional | CLI reference |
| `models/scene_mlp.joblib` | Optional | Better scene filter; HSV fallback if missing |
| `data/`, `graphify-out/`, `.git/` | Exclude | Not needed on Colab |

### Video input

- Supply your own `.mp4` broadcast clip (no sample video ships with the repo).
- **Start with 1–3 minutes** for your first run.
- Upload to Drive or Colab and set `VIDEO_PATH` below.

### Disk estimate

Approximate sampled frames per scene:

`camera_samples_per_scene + scene_duration_sec × ocr_samples_per_sec`

Multiply by number of scenes (depends on cut frequency) to estimate JPEG count and Drive usage.

### Optional dependency extras

| Extra | Packages | Needed for |
|-------|----------|------------|
| embedding | torch, torchvision | Camera ensemble (Pro profile) |
| court | torch, torchvision | Phase B `geom_pose` (included in Pro install) |
| match | torch, kornia, LightGlue | Phase C `link` stage |
| ocr | rapidocr, onnxruntime | OCR + graphics_sparse stages |
| ocr-vlm | openai | `enable_vlm=True` |
| viz | fastapi, uvicorn, python-multipart | Timeline web UI |

### Secrets

- `OPENAI_API_KEY` — only if `enable_vlm=True`. Set via Colab **Secrets** (`userdata`) or `os.environ`.


## Runtime profiles

Choose `RUNTIME_PROFILE` before running setup.

| Profile | Camera ID phases | Notes |
|---------|------------------|-------|
| **free** | Phase A only (HSV `scene_cluster`) | Disables B/C/D; no torch/LightGlue required |
| **pro** | Phases A–D full stack | Ensemble + geom_pose + link + graphics refine |

> **Preflight quirk:** `preflight()` gates torch on `ensemble_method == "ensemble"`, **not** on `fast_cameras`. The free profile sets both `fast_cameras=True` and `ensemble_method="hsv"`, and disables `cut_link_enabled` / graphics flags so full `STAGE_ORDER` preflight passes without LightGlue.


In [ ]:
# --- User configuration (edit these paths) ---
from pathlib import Path

RUNTIME_PROFILE = "free"  # "free" | "pro"

PROFILES = {
    "free": dict(
        fast_cameras=True,
        ensemble_method="hsv",
        geom_pose_enabled=False,
        persist_geom_pose_features=False,
        cut_link_enabled=False,
        graphics_sparse_enabled=False,
        graphics_provisional_refine_enabled=False,
        graphics_refine_enabled=False,
        ocr_scale=1.25,
        ocr_samples_per_sec=1.0,
        camera_samples_per_scene=3,
        enable_vlm=False,
        accelerator="cpu",
    ),
    "pro": dict(
        fast_cameras=False,
        ensemble_method="ensemble",
        geom_pose_enabled=True,
        persist_geom_pose_features=True,
        cut_link_enabled=True,
        graphics_sparse_enabled=True,
        graphics_provisional_refine_enabled=True,
        graphics_refine_enabled=True,
        ocr_scale=1.5,
        ocr_samples_per_sec=2.0,
        camera_samples_per_scene=5,
        enable_vlm=False,
        accelerator="auto",
    ),
}

# After unzipping the project zip:
ROOT = Path("/content/CV_problem")

# Video and output — use Drive paths for persistence across disconnects:
VIDEO_PATH = Path("/content/drive/MyDrive/videos/match.mp4")
OUTPUT_DIR = Path("/content/drive/MyDrive/pipeline_out")

print(f"Profile: {RUNTIME_PROFILE}")
print(f"ROOT: {ROOT}")
print(f"Video: {VIDEO_PATH}")
print(f"Output: {OUTPUT_DIR}")


## Environment setup

Run these cells once per Colab session.

In [ ]:
# Optional: check GPU (Pro runtime)
import shutil
if shutil.which("nvidia-smi"):
    !nvidia-smi
else:
    print("No GPU detected — free profile (hsv cameras) is appropriate.")


In [ ]:
# Mount Google Drive (recommended for persistence)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab — skip Drive mount.")


In [ ]:
# Unzip project — run after uploading zip to Drive or /content/
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/cv_problem.zip")  # edit to your zip path on Drive
FORCE_REEXTRACT = False  # set True to unzip again even if ROOT folder exists


def _project_ready(path: Path) -> bool:
    return (path / "src" / "broadcast_pipeline").is_dir()


def _resolve_project_root(base: Path) -> Path | None:
    """Find directory containing src/broadcast_pipeline (handles nested zip folders)."""
    if _project_ready(base):
        return base
    if not base.is_dir():
        return None
    for child in sorted(base.iterdir()):
        if child.is_dir() and _project_ready(child):
            return child
    return None


resolved = _resolve_project_root(ROOT)
if resolved is not None and not FORCE_REEXTRACT:
    if resolved != ROOT:
        print(f"Project code at {resolved} (nested inside {ROOT})")
    else:
        print(f"Project code found at {ROOT}")
    ROOT = resolved
elif ZIP_PATH.is_file():
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        print(f"Extracting {ZIP_PATH.name} ({len(zf.namelist())} entries) into {ROOT}")
        zf.extractall(ROOT)
    resolved = _resolve_project_root(ROOT)
    if resolved is None:
        raise FileNotFoundError(
            f"Zip extracted to {ROOT} but src/broadcast_pipeline not found. "
            "Re-zip with src/ at the top level, or set ROOT to the nested folder."
        )
    if resolved != ROOT:
        print(f"Project code at {resolved} (update ROOT in config cell if needed)")
    ROOT = resolved
    print(f"Ready: {ROOT / 'src' / 'broadcast_pipeline'}")
else:
    raise FileNotFoundError(
        f"No project at {ROOT} and zip not found at {ZIP_PATH}. "
        "Upload zip, set ZIP_PATH, or copy src/ into ROOT."
    )


In [ ]:
import sys

# Re-resolve in case kernel was restarted after unzip cell
if "_resolve_project_root" in globals():
    _found = _resolve_project_root(ROOT)
    if _found is not None:
        ROOT = _found

_marker = ROOT / "src" / "broadcast_pipeline"
assert _marker.is_dir(), (
    f"Missing {_marker}. Run the unzip cell above or set ROOT to the folder containing src/."
)
for entry in (str(ROOT), str(ROOT / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)
print(f"ROOT={ROOT}")
print("sys.path (head):", sys.path[:4])


In [ ]:
# Install dependencies (direct pip — avoids pyproject python>=3.13 constraint)
import subprocess
import sys

%pip install -q opencv-python scenedetect pandas scikit-learn scipy hdbscan matplotlib numpy joblib

if RUNTIME_PROFILE == "pro":
    # extra-index-url keeps PyPI primary; --index-url would break kornia/rapidocr on Colab
    %pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121
    %pip install -q certifi kornia --index-url https://pypi.org/simple
    %pip install -q 'git+https://github.com/cvg/LightGlue.git'
    %pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null || true
    %pip install -q onnxruntime-gpu --index-url https://pypi.org/simple
    %pip install -q 'rapidocr>=3.8' --index-url https://pypi.org/simple
    dl = ROOT / "scripts" / "download_lightglue_weights.py"
    if dl.is_file():
        subprocess.run([sys.executable, str(dl)], cwd=str(ROOT), check=False)
    else:
        print(f"Warning: missing {dl.name} — link stage may download weights at runtime")
    court_dl = ROOT / "scripts" / "download_tennis_court_model.py"
    if court_dl.is_file():
        subprocess.run([sys.executable, str(court_dl)], cwd=str(ROOT), check=False)
else:
    %pip install -q 'rapidocr>=3.8' onnxruntime --index-url https://pypi.org/simple

import importlib
importlib.invalidate_caches()
print(f"Install complete (profile={RUNTIME_PROFILE}).")


In [ ]:
# Verify imports
import importlib
import importlib.metadata

modules = [
    "broadcast_pipeline",
    "broadcast_pipeline.orchestrator",
    "broadcast_pipeline.cut_boundary_linker",
    "broadcast_pipeline.graphics_runner",
    "src.scene_ocr.extractor",
    "src.camera_assignemnt.scene_cluster.pipeline",
]
for name in modules:
    importlib.import_module(name)
    print(f"OK  {name}")

from src.camera_assignemnt.cut_match.config import cut_match_available

match_ok, match_reason = cut_match_available()
if RUNTIME_PROFILE == "pro":
    if match_ok:
        print("OK  cut_match (kornia + LightGlue)")
    else:
        raise RuntimeError(
            f"Pro profile requires cut_match stack: {match_reason}. "
            "Re-run the install cell above."
        )
else:
    print(f"cut_match: skipped (free profile)" + (f" — {match_reason}" if not match_ok else ""))

import pandas as pd
import cv2


def _rapidocr_distribution():
    for dist in ("rapidocr", "rapidocr-onnxruntime"):
        try:
            return dist, importlib.metadata.version(dist)
        except importlib.metadata.PackageNotFoundError:
            continue
    return None, None


ocr_dist, rapidocr_ver = _rapidocr_distribution()
if rapidocr_ver is None:
    raise RuntimeError(
        "rapidocr is not installed. Re-run the install cell above "
        "(expects: pip install 'rapidocr>=3.8')."
    )
print(f"pandas {pd.__version__}, opencv {cv2.__version__}, {ocr_dist} {rapidocr_ver}")

from rapidocr import RapidOCR

_probe = RapidOCR()
required = ("use_det", "text_det", "text_rec", "use_cls")
missing = [name for name in required if not hasattr(_probe, name)]
if missing:
    raise RuntimeError(
        f"{ocr_dist} {rapidocr_ver} is incompatible. "
        f"Missing RapidOCR attributes: {missing}. Install rapidocr>=3.8."
    )
print("OK  RapidOCR API (text_det / text_rec present)")
del _probe

from src.accelerator.install import check_gpu_stack, gpu_install_hints

gpu_report = check_gpu_stack("auto")
print(gpu_report)
if gpu_report.ocr_cuda_missing or gpu_report.torch_cuda_missing:
    print("GPU install hints:")
    print("\n".join(gpu_install_hints(gpu_report)))


## Configuration

`PipelineConfig` (`src/broadcast_pipeline/config.py`) controls every stage.

| Field | Default | Stage(s) | Description |
|-------|---------|----------|-------------|
| `video_path` | — | meta, extract | Input `.mp4` |
| `output_dir` | `data/pipeline` | all | Artifact root |
| `reference_csv` | `{output_dir}/approved_text_reference.csv` | reference, associate | Approved text list |
| `detector_threshold` | `27.0` | extract | PySceneDetect cut sensitivity (lower = more scenes) |
| `camera_samples_per_scene` | `5` | extract, cameras | Frames sampled per scene for clustering |
| `ocr_samples_per_sec` | `2.0` | extract | OCR frame sampling rate |
| `camera_approach` | `"scene_cluster"` | cameras | `"scene_cluster"` (default) or legacy `"embedding_cluster"` |
| `aggregate_camera_frames` | `True` | cameras | Phase A: one feature vector per scene |
| `stratify_by_scene_type` | `True` | cameras | Phase A: closeups abstain from global clustering |
| `scene_temporal_window` | `3` | cameras | Phase A: scene-timeline smoothing window |
| `scene_temporal_min_consensus` | `2` | cameras | Phase A: min neighbors for temporal fill |
| `ensemble_method` | `"ensemble"` | cameras | `"hsv"`, `"embedding"`, or `"ensemble"` |
| `fast_cameras` | `False` | cameras | Force HSV-only when `True` |
| `geom_pose_enabled` | `True` | cameras | Phase B: homography ensemble member |
| `persist_geom_pose_features` | `True` | cameras | Phase B: write `geom_pose_features.csv` |
| `cut_link_enabled` | `True` | link | Phase C: enable cut-boundary linking |
| `cut_match_min_confidence` | `0.5` | link | Min match confidence to accept link |
| `cut_match_high_confidence` | `0.75` | link | High-confidence threshold |
| `graphics_sparse_enabled` | `True` | graphics_sparse | Phase D: sparse bbox detection |
| `graphics_provisional_refine_enabled` | `True` | graphics_sparse | Phase D: early provisional refine |
| `graphics_refine_enabled` | `True` | graphics_profile | Phase D: layout fingerprint refine |
| `graphics_refine_min_confidence` | `0.6` | graphics_profile | Min confidence for graphics refine |
| `ocr_scale` | `1.5` | ocr | Upscale factor before OCR (lower = less RAM) |
| `ocr_preprocess` | `True` | ocr | CLAHE + preprocessing |
| `enable_vlm` | `False` | ocr | Escalate unread text to OpenAI VLM |
| `unk_token` | `"UNK"` | ocr, associate | Placeholder when text unreadable |
| `association_min_score` | `0.6` | associate | Min LCS score for partial match |
| `association_min_match_chars` | `3` | associate | Min LCS length |
| `enrich_enabled` | `True` | enrich | Temporal stabilization on/off |
| `enrich_region_iou` | `0.3` | enrich | IoU threshold for region tracking |
| `readability_size_multiplier` | `1.25` | enrich, aggregate | Readability area threshold |
| `default_new_text_approved` | `True` | reference | Auto-approve discovered tokens |
| `from_step` | `"all"` | — | Start stage for `run_pipeline` |
| `resume` | `False` | all | Skip stages whose outputs already exist |

`video_meta.json` stores the **absolute** video path from probe time. Harmless on Colab unless you move the video file.


In [ ]:
from broadcast_pipeline.config import PipelineConfig, STAGE_ORDER
from broadcast_pipeline.orchestrator import run_pipeline
from broadcast_pipeline.preflight import preflight

config = PipelineConfig(
    video_path=VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    resume=True,
    from_step="all",
    camera_approach="scene_cluster",
    aggregate_camera_frames=True,
    stratify_by_scene_type=True,
    scene_temporal_window=3,
    scene_temporal_min_consensus=2,
    **PROFILES[RUNTIME_PROFILE],
)
config.output_dir.mkdir(parents=True, exist_ok=True)

# Preflight only the stages that will actually run for this profile
preflight(config, STAGE_ORDER)
print("Preflight OK — all dependencies available.")
print(f"Stages: {' → '.join(STAGE_ORDER)}")
for key in (
    "camera_approach", "geom_pose_enabled", "cut_link_enabled",
    "graphics_sparse_enabled", "graphics_refine_enabled",
    "fast_cameras", "ensemble_method",
):
    print(f"  {key} = {getattr(config, key)!r}")


## Stage-by-stage deep dive

Each subsection explains one pipeline stage before you run it.

### Stage 1: `meta` — Video metadata

**Module:** `src/broadcast_pipeline/video_meta.py`

**Purpose:** Probe the input video for FPS, frame count, duration, and resolution.

**Output:** `video_meta.json`

**Colab notes:** Lightweight, seconds to run. No GPU required.

### Stage 2: `extract` — Scene detection & frame sampling

**Module:** `src/broadcast_pipeline/scene_extractor.py`

**Purpose:** Detect scene cuts, sample frames per scene for camera clustering and OCR, write JPEGs.

**Outputs:** `scenes.json`, `frame_index.csv`, `frames/scene_{id}_frame_{num}.jpg`

**Colab notes:** Full sequential video scan. Lower `ocr_samples_per_sec` to cut disk and downstream OCR cost.

### Stage 3: `filter` — Scene type classification

**Module:** `src/broadcast_pipeline/scene_filter.py` → `src/camera_assignemnt/scene_classifier/classifier.py`

**Purpose:** Classify each scene as `full_court` or `closeup` via majority vote over camera-sample frames.

**Output:** `scene_types.csv`

**Colab notes:** Fast, CPU-only. **Consumed by the cameras stage** — `stratify_by_scene_type=True` routes `full_court` scenes to global clustering and abstains closeups (Phase A).

### Stage 4: `cameras` — 4-phase camera identity (Phases A + B)

**Module:** `src/broadcast_pipeline/camera_assignment.py` → `src/camera_assignemnt/scene_cluster/assignment.py`

**Purpose:** Assign `camera_id` per scene via scene-level unsupervised clustering.

**Algorithm (Phase A):**
- `aggregate_samples_per_scene()` — one feature vector per scene
- `stratify_by_scene_type` — closeups abstain (`unknown`); only `full_court` enters global clustering
- `scene_temporal_fill()` — temporal smoothing on the scene timeline

**Algorithm (Phase B, Pro):**
- Fourth ensemble member `geom_pose` — homography-derived viewpoint descriptors
- Ensemble: HSV + ResNet50 + DINOv2 + geom_pose co-association vote

**Outputs:** `scene_assignments.csv`, `frame_assignments.csv`, optional `geom_pose_features.csv`

**Colab notes:** Heaviest GPU stage in Pro profile. First run downloads torchvision + DINOv2 weights.

### Stage 5: `link` — Cut-boundary linking (Phase C)

**Module:** `src/broadcast_pipeline/cut_boundary_linker.py` → `src/camera_assignemnt/cut_match/`

**Purpose:** Propagate `camera_id` from outgoing `full_court` to incoming `closeup` at scene cuts using static-region feature matching (LightGlue + SuperPoint).

**Output:** `cut_boundary_matches.csv`; updates `scene_assignments` / `frame_assignments` with `assignment_source=cut_link`

**Colab notes:** Pro profile only. Requires kornia + LightGlue. Pre-download weights via `scripts/download_lightglue_weights.py`. Closeups still `unknown` after cameras alone — this is expected until link runs.

### Stage 6: `graphics_sparse` — Sparse graphics detection (Phase D pass 1)

**Module:** `src/broadcast_pipeline/graphics_runner.py` → `src/scene_ocr/extractor.py` (`detect_boxes`)

**Purpose:** Run detection-only text-box finding on camera-sample frames (no recognition). Optional provisional refine for high-confidence layout matches.

**Output:** `frame_graphics_sparse.csv`

**Colab notes:** Pro profile. Uses RapidOCR detector. Runs **before** OCR so downstream stages inherit better `camera_id` on closeups.

### Stage 7: `ocr` — On-screen text extraction

**Module:** `src/broadcast_pipeline/ocr_runner.py` → `src/scene_ocr/`

**Purpose:** Run RapidOCR on OCR-sample frames, assess readability, optionally escalate to VLM.

**Output:** `frame_ocr.csv`

**Colab notes:** Frame-level resume via `ocr_done_keys()`. OCR now runs after link/graphics_sparse — `camera_id` on closeup frames is more accurate.

### Stage 8: `reference` — Text reference discovery

**Module:** `src/broadcast_pipeline/text_reference.py`

**Output:** `approved_text_reference.csv`

**Colab notes:** **Human-in-the-loop checkpoint.** Review before associate.

### Stage 9: `enrich` — Temporal OCR stabilization

**Module:** `src/broadcast_pipeline/text_enrich.py`

**Output:** `frame_ocr_enriched.csv`

### Stage 10: `graphics_profile` — Graphics fingerprint refine (Phase D pass 2)

**Module:** `src/broadcast_pipeline/graphics_profile.py`

**Purpose:** Build per-camera graphics layout profiles from enriched OCR + sparse detections. Refine residual `unknown` / split-vote scenes via histogram match.

**Outputs:** `camera_graphics_profile.json`; updated `scene_assignments` with `assignment_source=graphics_refine`

**Colab notes:** Requires enrich to complete first.

### Stage 11: `associate` — Map OCR to reference text

**Module:** `src/broadcast_pipeline/text_associate.py`

**Outputs:** `frame_text_associated.csv`, `dropped_text.csv`

### Stage 12: `aggregate` — Per-camera text timelines

**Module:** `src/broadcast_pipeline/aggregator.py`

**Outputs:** `aggregated_complete.csv`, `aggregated_partial.csv`, `pipeline_summary.json`

## Chunked execution

`run_pipeline()` has **no built-in `until_step`**. Setting `from_step="meta"` runs through `aggregate` in one call.

This notebook defines `run_stages(config, from_step, until_step)` which temporarily limits the stage list, enabling Colab-friendly chunks.

### Resume semantics

| Mechanism | Granularity | When |
|-----------|-------------|------|
| `config.resume=True` | Stage | Skips stage if output file exists |
| OCR `ocr_done_keys()` | Frame | Skips frames already in `frame_ocr.csv` |
| Human-loop re-run | Stage | After editing reference — use `resume=False` or delete downstream CSVs |

> **Trap:** After editing `approved_text_reference.csv`, `resume=True` **skips** associate/aggregate if `frame_text_associated.csv` exists. See [Human-in-the-loop](#human-in-the-loop) below.


In [ ]:
import broadcast_pipeline.artifacts as artifacts_mod
import broadcast_pipeline.orchestrator as orchestrator_mod
from broadcast_pipeline.config import STAGE_ORDER, PipelineConfig
from broadcast_pipeline.orchestrator import run_pipeline
from broadcast_pipeline.types import PipelineSummary


def run_stages(
    config: PipelineConfig,
    from_step: str,
    until_step: str,
) -> PipelineSummary:
    """Run pipeline stages from *from_step* through *until_step* (inclusive)."""
    start = STAGE_ORDER.index(from_step)
    end = STAGE_ORDER.index(until_step)
    slice_stages = STAGE_ORDER[start : end + 1]

    original_artifacts = artifacts_mod.stages_from
    original_orchestrator = orchestrator_mod.stages_from

    def limited_stages_from(step: str):
        if step == "all":
            return slice_stages
        base_start = STAGE_ORDER.index(step)
        full = STAGE_ORDER[base_start:]
        return tuple(s for s in full if s in slice_stages)

    saved_from = config.from_step
    print(f"run_stages: {' → '.join(slice_stages)}")
    try:
        artifacts_mod.stages_from = limited_stages_from
        orchestrator_mod.stages_from = limited_stages_from
        config.from_step = from_step
        return run_pipeline(config)
    finally:
        artifacts_mod.stages_from = original_artifacts
        orchestrator_mod.stages_from = original_orchestrator
        config.from_step = saved_from


In [ ]:
import ast
import json
from collections import Counter
from IPython.display import display
import pandas as pd


def attach_scene_type(df, types_df):
    base = df.drop(
        columns=[c for c in df.columns if c == "scene_type" or c.startswith("scene_type_")],
        errors="ignore",
    )
    st = types_df[["scene_id", "scene_type"]].copy()
    st["scene_id"] = st["scene_id"].astype(int)
    base = base.copy()
    base["scene_id"] = base["scene_id"].astype(int)
    return base.merge(st, on="scene_id", how="left")


def split_vote_scenes(df):
    scenes = []
    for _, row in df.iterrows():
        raw = row.get("camera_vote_counts_json", "")
        if not isinstance(raw, str) or not raw.strip():
            continue
        try:
            counts = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            continue
        if not isinstance(counts, dict):
            continue
        leaders = [k for k, v in counts.items() if v == max(counts.values())]
        if len(leaders) > 1:
            scenes.append(int(row["scene_id"]))
    return scenes


def camera_id_metrics(scene_assignments, scene_types=None):
    df = scene_assignments.copy()
    if scene_types is not None and not scene_types.empty:
        df = attach_scene_type(df, scene_types)
    metrics = {
        "n_scenes": int(len(df)),
        "n_cameras": int(df.loc[df["camera_id"] != "unknown", "camera_id"].nunique()),
        "unknown_scene_rate": float((df["camera_id"] == "unknown").mean()),
        "split_vote_scenes": split_vote_scenes(df),
    }
    if "assignment_source" in df.columns:
        metrics["assignment_source_counts"] = (
            df["assignment_source"].fillna("cluster").value_counts().to_dict()
        )
    if "scene_type" in df.columns:
        by_type = df.groupby("scene_type").agg(
            scenes=("scene_id", "count"),
            unknown=("camera_id", lambda s: int((s == "unknown").sum())),
            cameras=("camera_id", lambda s: s[s != "unknown"].nunique()),
        )
        metrics["by_scene_type"] = by_type.reset_index().to_dict(orient="records")
    return metrics


def save_phase_summary(config, label, metrics):
    path = config.output_dir / f"phase_{label}_summary.json"
    path.write_text(json.dumps(metrics, indent=2))
    print(f"Saved {path}")
    print(json.dumps(metrics, indent=2))


### Group 1: `meta` → `extract`

In [ ]:
summary_g1 = run_stages(config, from_step="meta", until_step="extract")
print("Artifacts:", {k: str(v) for k, v in summary_g1.artifacts.items()})
print(f"Scenes: {summary_g1.n_scenes}")


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

meta = json.loads(config.artifact("video_meta").read_text())
print(f"Duration: {meta['duration_sec']:.1f}s, FPS: {meta['fps']:.2f}, "
      f"Frames: {meta['frame_count']}, Size: {meta['width']}x{meta['height']}")

scenes = json.loads(config.artifact("scenes").read_text())
print(f"Detected {len(scenes)} scene(s)")

frame_index = pd.read_csv(config.artifact("frame_index"))
print(frame_index.groupby("sample_role").size())

sample = frame_index.iloc[0]
if Path(sample["frame_path"]).is_file():
    display(Image(filename=str(sample["frame_path"]), width=400))


### Group 2: `filter` → `cameras` (Phases A + B)

Runs scene-level clustering with optional `geom_pose` ensemble member (Pro profile).

In [ ]:
assignments_before_cameras = None
if config.artifact("scene_assignments").is_file():
    assignments_before_cameras = pd.read_csv(config.artifact("scene_assignments"))

summary_g2 = run_stages(config, from_step="filter", until_step="cameras")
print(f"Cameras: {summary_g2.n_cameras}")


In [ ]:
scene_types = pd.read_csv(config.artifact("scene_types"))
scene_assignments = pd.read_csv(config.artifact("scene_assignments"))
frame_assignments = pd.read_csv(config.artifact("frame_assignments"))
frame_index = pd.read_csv(config.artifact("frame_index"))

camera_rows = frame_index[frame_index["sample_role"] == "camera"]
n_scenes = camera_rows["scene_id"].nunique()
n_camera_frames = len(camera_rows)
print(f"Clustering input: {n_scenes} scene vectors (from {n_camera_frames} camera frames)")

metrics_g2 = camera_id_metrics(scene_assignments, scene_types)
save_phase_summary(config, "ab_cameras", metrics_g2)
display(scene_assignments.head(12))

geom_path = config.artifact("geom_pose_features")
if geom_path.is_file():
    geom_df = pd.read_csv(geom_path)
    print(f"geom_pose_features: {len(geom_df)} rows")
    if "confidence" in geom_df.columns:
        print(f"  abstain (confidence=0): {(geom_df['confidence'] == 0).sum()}")
    display(geom_df.head(8))
elif config.geom_pose_enabled:
    print(f"No {geom_path.name} — set persist_geom_pose_features=True")
else:
    print("geom_pose disabled (free profile)")


### Phase A + B — improvement summary

| Signal | Effect |
|--------|--------|
| `aggregate_camera_frames` | One feature vector per scene — no split intra-scene votes |
| `stratify_by_scene_type` | Closeups abstain instead of polluting global clusters |
| `scene_temporal_fill` | Fills isolated unknowns on the scene timeline |
| `geom_pose` (Pro) | Geometry voter for `full_court` wide shots |

**Remaining gap after cameras alone:** closeup scenes may still be `unknown` until Phase C (`link`).

### Group 3: `link` (Phase C)

Pro profile only. Propagates `camera_id` across wide→closeup cuts.

In [ ]:
if config.cut_link_enabled:
    assignments_before_link = scene_assignments.copy()
    summary_g3 = run_stages(config, from_step="link", until_step="link")
    scene_assignments = pd.read_csv(config.artifact("scene_assignments"))
    frame_assignments = pd.read_csv(config.artifact("frame_assignments"))
    print(f"Link complete — artifacts: {list(summary_g3.artifacts.keys())}")
else:
    print("Phase C skipped — cut_link_enabled=False (free profile)")


In [ ]:
if config.cut_link_enabled:
    matches_path = config.artifact("cut_boundary_matches")
    if matches_path.is_file():
        matches = pd.read_csv(matches_path)
        print(f"cut_boundary_matches: {len(matches)} rows")
        if "accepted" in matches.columns:
            print(f"  accepted: {int(matches['accepted'].sum())}")
            display(matches[matches["accepted"]].head(10))
    metrics_before = camera_id_metrics(assignments_before_link, scene_types)
    metrics_after = camera_id_metrics(scene_assignments, scene_types)
    link_summary = {
        "before_link": metrics_before,
        "after_link": metrics_after,
        "delta_unknown_rate": metrics_after["unknown_scene_rate"] - metrics_before["unknown_scene_rate"],
    }
    save_phase_summary(config, "c_link", link_summary)
else:
    print("No link inspection — Phase C disabled")


### Phase C — improvement summary

Phase C targets **closeup unknowns** by matching static background at cut boundaries. Negative `delta_unknown_rate` in the link summary means fewer unknown scenes after linking.

### Group 4: `graphics_sparse` (Phase D pass 1)

Pro profile only. Detection-only graphics boxes on camera samples.

In [ ]:
if config.graphics_sparse_enabled:
    summary_g4 = run_stages(config, from_step="graphics_sparse", until_step="graphics_sparse")
    print(f"graphics_sparse complete — artifacts: {list(summary_g4.artifacts.keys())}")
else:
    print("Phase D pass 1 skipped — graphics_sparse_enabled=False (free profile)")


In [ ]:
if config.graphics_sparse_enabled:
    sparse_path = config.artifact("frame_graphics_sparse")
    if sparse_path.is_file():
        sparse_df = pd.read_csv(sparse_path)
        print(f"frame_graphics_sparse: {len(sparse_df)} rows")
        display(sparse_df.head(8))
    scene_assignments = pd.read_csv(config.artifact("scene_assignments"))
    if "assignment_source" in scene_assignments.columns:
        prov = scene_assignments[scene_assignments["assignment_source"].fillna("") != "cluster"]
        print(f"Non-cluster assignments after sparse: {len(prov)}")


### Group 5: `ocr`

Safe to re-run if interrupted — frame-level resume skips completed frames.

In [ ]:
import src.scene_ocr.extractor as _ocr_ext

_ocr_ext._get_engine.cache_clear()

summary_g5 = run_stages(config, from_step="ocr", until_step="ocr")
print(f"OCR frames: {summary_g5.n_frames}")


In [ ]:
frame_ocr = pd.read_csv(config.artifact("frame_ocr"))
print(f"Rows: {len(frame_ocr)}, columns: {list(frame_ocr.columns)}")
if len(frame_ocr):
    print(frame_ocr[["scene_id", "frame_number", "camera_id", "verdict", "used_unk"]].head())


### Group 6: `reference` — pause for review

`default_new_text_approved=True` means discovered tokens are auto-approved. Set `approved=false` for spurious entries before Group 7.

In [ ]:
summary_g6 = run_stages(config, from_step="reference", until_step="reference")
ref = pd.read_csv(config.reference_csv)
print(f"Reference entries: {len(ref)} ({ref['approved'].sum()} approved)")
ref.head(10)


### Group 7: `enrich` → `graphics_profile` → `associate` → `aggregate`

Phase D pass 2 (graphics refine) runs inside `graphics_profile` after enrich.

In [ ]:
assignments_before_graphics = pd.read_csv(config.artifact("scene_assignments"))

summary_g7 = run_stages(config, from_step="enrich", until_step="aggregate")
print("Pipeline complete.")
for name, path in summary_g7.artifacts.items():
    print(f"  {name}: {path}")


In [ ]:
scene_assignments = pd.read_csv(config.artifact("scene_assignments"))
metrics_final = camera_id_metrics(scene_assignments, scene_types)
save_phase_summary(config, "d_final", metrics_final)

profile_path = config.artifact("camera_graphics_profile")
if profile_path.is_file():
    graphics_profile = json.loads(profile_path.read_text())
    print(f"camera_graphics_profile: {len(graphics_profile)} camera(s)")
    for cam_id, prof in list(graphics_profile.items())[:5]:
        print(f"  {cam_id}: {prof.get('n_observations', '?')} observations")

if config.graphics_refine_enabled:
    refined = scene_assignments[
        scene_assignments.get("assignment_source", "") == "graphics_refine"
    ]
    print(f"graphics_refine assignments: {len(refined)}")
    if len(refined):
        cols = [c for c in ["scene_id", "camera_id", "assignment_source", "scene_type"] if c in refined.columns]
        display(refined[cols].head(10))
    before = camera_id_metrics(assignments_before_graphics, scene_types)
    after = camera_id_metrics(scene_assignments, scene_types)
    gfx_summary = {
        "before_graphics": before,
        "after_graphics": after,
        "delta_unknown_rate": after["unknown_scene_rate"] - before["unknown_scene_rate"],
    }
    save_phase_summary(config, "d_graphics_refine", gfx_summary)


### Phase D — improvement summary

Phase D targets **residual unknowns and split-vote scenes** using broadcast graphics layout fingerprints. Compare `delta_unknown_rate` in `phase_d_graphics_refine_summary.json`.

## Human-in-the-loop

After editing `approved_text_reference.csv`:

1. Set `config.from_step = "associate"`
2. Set `config.resume = False` (**required** — otherwise associate is skipped)
3. Run `run_pipeline(config)` or `run_stages(config, "associate", "aggregate")`

Alternatively delete: `frame_text_associated.csv`, `dropped_text.csv`, `aggregated_complete.csv`, `aggregated_partial.csv`, `pipeline_summary.json`


In [ ]:
# Uncomment after editing approved_text_reference.csv:
# config.from_step = "associate"
# config.resume = False
# summary_rerun = run_stages(config, from_step="associate", until_step="aggregate")


## Full end-to-end run

Alternative to chunked execution — one cell runs everything (best on Pro with short clips).

In [ ]:
# config.from_step = "all"
# config.resume = True
# summary = run_pipeline(config)
# print(f"Scenes={summary.n_scenes}, Cameras={summary.n_cameras}, OCR frames={summary.n_frames}")


## Results exploration

### Aggregated timeline columns

| Column | Meaning |
|--------|---------|
| `camera_id` | Clustered camera identifier |
| `text` | Observed text (complete or partial) |
| `text_kind` | `complete` or `partial` |
| `mapped_complete_text` | Reference text for partial matches |
| `total_duration_sec` | Weighted time on screen |
| `frame_ranges` | Semicolon-separated frame ranges |
| `n_frames_present` | Frames with this text |
| `n_frames_good` / `n_frames_partial` | Readability breakdown |
| `n_frames_enriched` | Frames where enrich was applied |
| `dominant_readability` | Majority readability label |


In [ ]:
import json
import pandas as pd
from IPython.display import display

complete = pd.read_csv(config.artifact("aggregated_complete"))
partial = pd.read_csv(config.artifact("aggregated_partial"))
dropped = pd.read_csv(config.artifact("dropped_text"))
summary_json = json.loads(config.artifact("pipeline_summary").read_text())

print("=== Pipeline summary ===")
for k, v in summary_json.items():
    print(f"  {k}: {v}")

print(f"\nComplete rows: {len(complete)}, Partial rows: {len(partial)}, Dropped: {len(dropped)}")
if len(complete):
    display(complete.head(10))
if len(dropped):
    print("\nDropped text (sample):")
    display(dropped.head(5))


In [ ]:
# Timeline bar chart per camera (matplotlib — reliable on Colab)
import matplotlib.pyplot as plt

if len(complete):
    fig, ax = plt.subplots(figsize=(12, max(3, complete["camera_id"].nunique() * 0.5)))
    cameras = sorted(complete["camera_id"].unique())
    y_map = {c: i for i, c in enumerate(cameras)}
    for _, row in complete.iterrows():
        y = y_map[row["camera_id"]]
        ax.barh(y, row["total_duration_sec"], left=0, height=0.6, alpha=0.7)
        ax.text(0.1, y, str(row["text"])[:30], va="center", fontsize=8)
    ax.set_yticks(range(len(cameras)))
    ax.set_yticklabels(cameras)
    ax.set_xlabel("Duration (sec)")
    ax.set_title("Complete text timeline by camera")
    plt.tight_layout()
    plt.show()
else:
    print("No complete rows to chart.")


## Optional timeline web UI

Requires `pip install fastapi uvicorn python-multipart` and `static/timeline_viz/` in your zip.

**CORS limitation:** `create_app()` only allows `localhost` origins. On Colab, use `serve_kernel_port_as_window` (iframe) rather than opening the raw proxy URL.


In [ ]:
# %pip install -q fastapi uvicorn python-multipart

# import threading
# import uvicorn
# from broadcast_pipeline.viz.server import create_app

# app = create_app(OUTPUT_DIR.resolve(), (ROOT / "static" / "timeline_viz").resolve())

# def _serve():
#     uvicorn.run(app, host="0.0.0.0", port=8765, log_level="warning")

# threading.Thread(target=_serve, daemon=True).start()

# try:
#     from google.colab import output
#     output.serve_kernel_port_as_window(8765, path="/", height=800)
# except ImportError:
#     print("Open http://127.0.0.1:8765 in a local browser")


## Standalone OCR debug

For single-frame OCR outside the full pipeline, use `scripts/run_ocr.py`:

```bash
python scripts/run_ocr.py --image path/to/frame.jpg --assess --json
```

Install: `pip install 'rapidocr>=3.8'` (+ `onnxruntime` or `onnxruntime-gpu`; `openai` for `--openai-vlm`).


## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `ImportError: src.camera_assignemnt` | Missing ROOT on sys.path | Add repo root: `sys.path.insert(0, str(ROOT))` |
| `cut_match: ...` preflight error | kornia not on torch index | Re-run install cell; or `%pip install kornia --index-url https://pypi.org/simple` |
| `geom_pose` abstains all scenes | Missing court model | Run `scripts/download_tennis_court_model.py`; ensure `full_court` scenes |
| Closeups `unknown` after cameras | Expected before link | Run Group 3 (`link` stage, Phase C) |
| `graphics_refine` no changes | Enrich not complete | Run Group 7; lower `graphics_refine_min_confidence` for debug |
| CUDA OOM on cameras | Ensemble on small GPU | Use free profile or lower `camera_samples_per_scene` |
| OCR slow / RAM spikes | High `ocr_scale` or `ocr_samples_per_sec` | Lower both in profile |
| Empty associated text | Unapproved reference entries | Set `approved=true` in reference CSV |
| Session disconnect | Long video | Drive output + `resume=True` + chunked `run_stages` |
| Missing rapidocr | OCR not installed or wrong wheel | Re-run install cell: `pip install 'rapidocr>=3.8'` (+ onnxruntime) |
| ROOT exists but import fails | Empty/wrong folder | Unzip cell checks `src/broadcast_pipeline`; set `FORCE_REEXTRACT=True` |
| torch required on free profile | Phase flags not disabled | Free profile must set `cut_link_enabled=False` and `ensemble_method="hsv"` |
| Associate unchanged after CSV edit | `resume=True` skips existing outputs | `resume=False` or delete downstream CSVs |
| Viz UI loads but API fails | Colab CORS | Use `serve_kernel_port_as_window` |
| First cameras run slow | Model download | Needs network; one-time per session |
| OCR still on CPU with GPU runtime | CPU `onnxruntime` installed | `%pip uninstall -y onnxruntime onnxruntime-gpu && %pip install onnxruntime-gpu` |


## Out of scope

The following exist in the repo but are **not** the primary focus of this notebook:

- `src/camera_split_segment.py` — legacy frame-extraction helper
- `scripts/train_scene_classifier.py`, `scripts/run_camera_clustering.py` — offline model training
- `scripts/eval_camera_clustering.py`, `scripts/eval_cut_boundary_linking.py`, `scripts/eval_homography.py` — offline evaluation (optional; run when GT labels exist)
- Legacy `camera_approach="embedding_cluster"` — per-frame clustering path kept for comparison
